# Cascade v5 — Google Colab

1. Runtime: **GPU** (menu Runtime → Change runtime type → T4 GPU)
2. Clone/upload repo ke `/content/cascade-v5-architecture`
3. Jalankan sel berurutan

LSTM: CUDA · LGBM: CPU (stabil di Colab) · MODAL = **5.0 USD**

## 1. Clone repo

Ganti `REPO_URL` dengan URL Git Anda, atau upload zip lalu unzip ke `/content/cascade-v5-architecture`.

In [ ]:
# Opsional: clone dari GitHub (edit URL)
REPO_URL = ""  # contoh: https://github.com/USER/cascade-v5-architecture.git
ROOT = "/content/cascade-v5-architecture"

if REPO_URL:
    !git clone {REPO_URL} {ROOT}
else:
    print("Upload folder repo ke", ROOT, "lalu lanjut ke sel berikutnya.")

## 2. Bootstrap Colab

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/content/cascade-v5-architecture")
%cd {ROOT}

sys.path.insert(0, str(ROOT))
from tools.colab_bootstrap import setup_colab, run

setup_colab(repo_root=ROOT, mount_drive=False, install_deps=True)

from config import COLAB_QUICK_COINS
COINS = " ".join(COLAB_QUICK_COINS)  # pilot 3 koin
print("COINS =", COINS)

## 3. Pipeline (pilot 3 koin)

Urutan wajib. Fetch bisa 30–60+ menit untuk 3 koin; 20 koin jauh lebih lama.

In [ ]:
run(f"python pipeline/01_fetch.py --coins {COINS}")
run(f"python pipeline/02_clean.py --coins {COINS}")
run(f"python pipeline/03_engineer.py --coins {COINS}")

In [ ]:
run(f"python pipeline/04_train_lgbm.py --all")  # gabung semua koin yang sudah di-engineer
run(f"python pipeline/05a_momentum_labels.py --coins {COINS}")
run(f"python pipeline/05b_build_sequences.py --coins {COINS}")
run("python pipeline/05d_oof_residuals.py --all")
run("python pipeline/05c_train_momentum_expert.py --all")
run(f"python pipeline/06_train_guardian.py --coins {COINS}")

## 4. Holdout (opsional)

Jalankan fetch/clean/engineer holdout dulu, lalu backtest.

In [ ]:
# run("python pipeline/01_fetch.py --all --holdout")
# run("python pipeline/02_clean.py --all --holdout")
# run("python pipeline/03_engineer.py --all --holdout")
run(f"python pipeline/07_holdout_backtest.py --coins {COINS}")

## 5. Laporan overfitting + simpan ke Drive (opsional)

In [ ]:
run("python tools/overfitting_report.py")
run("python tools/benchmark_plan.py")

# Opsional: salin models/ dan reports/ ke Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r models reports /content/drive/MyDrive/cascade_v5_run/